# Fixed Income Trading Analytics
**Rates Dashboard — Interactive Notebook**

This notebook imports the `fixed_income` Python library and lets you run:
- Z-score analysis on swap rates and spreads
- Carry & rolldown calculations
- Trade setup (outrights, spreads, butterflies)
- Bond analytics (DV01, convexity, asset swaps)
- Spread option pricing
- Wedge analysis
- Swaption expected returns
- Portfolio analytics (beta, efficient frontier, Sharpe)
- Weekly update tables (like the OneNote output)

---
**All data is pulled live from FRED** using the same pipeline as the Streamlit dashboard.

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.expanduser('~/rates-dashboard'))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Import the fixed income library
import fixed_income as fi

print(f'fixed_income loaded — {len(fi.__all__)} functions available')

In [ ]:
# Load live data from the dashboard pipeline
from data.pipeline import DataPipeline

pipeline = DataPipeline()
df = pipeline.get_data()
print(f'Data loaded: {len(df)} rows  |  {df.index[0].date()} → {df.index[-1].date()}')
print(f'Columns: {list(df.columns)}')

In [ ]:
# ── Convenience: current curve snapshot and overnight rate ──────────────────
from config import TREASURY_TENORS, SOFR_SERIES

TENOR_COLS = {
    '2Y': 'DGS2', '3Y': 'DGS3', '5Y': 'DGS5',
    '7Y': 'DGS7', '10Y': 'DGS10', '20Y': 'DGS20', '30Y': 'DGS30'
}

def get_curve_snapshot(df, date=None):
    """Current (or historical) par curve as a dict {tenor: rate_%}."""
    row = df.loc[date] if date else df.dropna(subset=list(TENOR_COLS.values()), how='all').iloc[-1]
    return {t: float(row[col]) for t, col in TENOR_COLS.items() if col in df.columns and not pd.isna(row[col])}

def get_rate_series(df, tenor):
    """Get historical rate series for a tenor label."""
    col = TENOR_COLS.get(tenor)
    if col and col in df.columns:
        return df[col].dropna()
    return pd.Series(dtype=float)

curve = get_curve_snapshot(df)
# SOFR as overnight proxy (or use Fed Funds if not available)
sofr_col = 'SOFR' if 'SOFR' in df.columns else 'DFF'
on_rate = float(df[sofr_col].dropna().iloc[-1]) if sofr_col in df.columns else 5.3

print('Current curve:')
for t, r in curve.items():
    print(f'  {t}: {r:.3f}%')
print(f'Overnight rate: {on_rate:.3f}%')

---
## 1. Z-Score Analysis
How cheap/rich are current rates relative to their own history?

In [ ]:
# ── Z-score summary table for all tenors ───────────────────────────────────
rate_df = pd.concat({t: df[col] for t, col in TENOR_COLS.items() if col in df.columns}, axis=1).dropna(how='all')

z_table = fi.zscore_table(rate_df, window=252)
fi.style_zscore_table(z_table)

In [ ]:
# ── Rolling z-score chart for a chosen tenor ───────────────────────────────
TENOR = '10Y'   # ← change this
WINDOW = 252    # ← 1Y lookback

s = get_rate_series(df, TENOR)
z = fi.zscore(s, WINDOW)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=[f'{TENOR} Rate (%)', f'{TENOR} Z-score ({WINDOW}d)'])
fig.add_trace(go.Scatter(x=s.index, y=s.values, name='Rate', line=dict(color='#4fc3f7')), row=1, col=1)
fig.add_trace(go.Scatter(x=z.index, y=z.values, name='Z-score', line=dict(color='#ff8a65')), row=2, col=1)
fig.add_hline(y=1.5, line_dash='dash', line_color='green', row=2, col=1)
fig.add_hline(y=-1.5, line_dash='dash', line_color='red', row=2, col=1)
fig.add_hline(y=0, line_dash='dot', line_color='grey', row=2, col=1)
fig.update_layout(template='plotly_dark', height=500, showlegend=False)
fig.show()

In [ ]:
# ── Spread z-scores ────────────────────────────────────────────────────────
spreads_to_check = [('2Y','10Y'), ('2Y','30Y'), ('5Y','30Y'), ('10Y','30Y')]

rows = []
for t1, t2 in spreads_to_check:
    s1 = get_rate_series(df, t1)
    s2 = get_rate_series(df, t2)
    spread = (s2 - s1).dropna() * 100  # bps
    if len(spread) < 50: continue
    rows.append({
        'Spread': f'{t1}/{t2}',
        'Current (bps)': round(float(spread.iloc[-1]), 1),
        '1Y ago (bps)':  round(float(spread.iloc[-253]), 1) if len(spread)>253 else None,
        'Chg 1Y (bps)':  round(float(spread.iloc[-1]-spread.iloc[-253]), 1) if len(spread)>253 else None,
        'Z-score (1Y)':  round(fi.zscore_current(spread, 252), 2),
        'Pctile (1Y)':   round(float(fi.percentile_rank(spread, 252).iloc[-1]), 1),
    })

pd.DataFrame(rows).set_index('Spread')

---
## 2. Carry & Rolldown
How much do you earn by holding a position for one month?

In [ ]:
# ── Carry + rolldown for all outrights ─────────────────────────────────────
rows = []
for tenor in ['2Y','3Y','5Y','7Y','10Y','20Y','30Y']:
    if tenor not in curve: continue
    cr = fi.snapshot_carry_rolldown(curve, on_rate, 'outright', tenor, holding_months=1)
    rows.append({
        'Tenor': tenor,
        'Rate (%)': round(curve[tenor], 3),
        'Carry (bps/m)': cr['carry'],
        'Rolldown (bps/m)': cr['rolldown'],
        'Total (bps/m)': cr['total'],
        'Ann (bps/yr)': round(cr['total']*12, 1),
    })

cr_df = pd.DataFrame(rows).set_index('Tenor')
fi.style_returns_table(cr_df[['Carry (bps/m)','Rolldown (bps/m)','Total (bps/m)','Ann (bps/yr)']])

In [ ]:
# ── Carry + rolldown for curve spreads ─────────────────────────────────────
rows = []
for t1, t2 in [('2Y','10Y'), ('2Y','30Y'), ('5Y','10Y'), ('5Y','30Y'), ('10Y','30Y')]:
    if t1 not in curve or t2 not in curve: continue
    cr = fi.snapshot_carry_rolldown(curve, on_rate, 'spread', t2, t1, holding_months=1)
    rows.append({
        'Spread': f'{t1}/{t2}',
        'Spread (bps)': round((curve[t2]-curve[t1])*100, 1),
        'Carry (bps/m)': cr['carry'],
        'Rolldown (bps/m)': cr['rolldown'],
        'Total (bps/m)': cr['total'],
        'Ann (bps/yr)': round(cr['total']*12, 1),
    })

cr_spreads = pd.DataFrame(rows).set_index('Spread')
fi.style_returns_table(cr_spreads[['Carry (bps/m)','Rolldown (bps/m)','Total (bps/m)','Ann (bps/yr)']])

---
## 3. Trade Setup — Build a Trade Book

In [ ]:
# ── Build a full trade book ─────────────────────────────────────────────────
rate_df_clean = rate_df.ffill(limit=3).dropna(subset=['2Y','10Y'])
on_series = (df[sofr_col] if sofr_col in df.columns else pd.Series(on_rate, index=df.index)).reindex(rate_df_clean.index, method='ffill')

book = fi.build_trade_book(
    rate_df       = rate_df_clean,
    overnight_series = on_series,
    outright_tenors  = ['2Y','5Y','10Y','30Y'],
    spread_pairs     = [('2Y','10Y'), ('5Y','30Y'), ('2Y','30Y')],
    fly_triplets     = [('2Y','5Y','10Y'), ('5Y','10Y','30Y')],
    curve_snapshot   = curve,
    zscore_window    = 252,
    holding_months   = 1,
)

print('── Outrights ──')
display(book['outrights'])
print('── Spreads ──')
display(book['spreads'])
print('── Butterflies ──')
display(book['flies'])

---
## 4. Bond Analytics
DV01, convexity, asset swap spread for a specific bond.

In [ ]:
# ── Quick bond analytics ────────────────────────────────────────────────────
# Edit these inputs for any bond:
COUPON       = 4.500   # annual coupon %
MATURITY_YRS = 10.0    # years to maturity
YTM          = curve.get('10Y', 4.8)  # yield to maturity %
NOTIONAL     = 10_000_000  # $10M

analytics = fi.quick_analytics(COUPON, MATURITY_YRS, YTM, NOTIONAL)
print('Bond Analytics:')
for k, v in analytics.items():
    print(f'  {k}: {v}')

In [ ]:
# ── DV01 across the curve ────────────────────────────────────────────────────
rows = []
for tenor, rate in curve.items():
    tenor_yr = fi.TENOR_YEARS.get(tenor)
    if tenor_yr is None: continue
    cfs = fi.bond_cashflows(rate, tenor_yr)
    dv01 = fi.dv01_bond(cfs, rate, 10_000_000)
    conv = fi.convexity(cfs, rate)
    rows.append({'Tenor': tenor, 'Rate (%)': round(rate,3),
                 'DV01 ($10M)': round(dv01,0), 'Convexity': round(conv,3)})

pd.DataFrame(rows).set_index('Tenor')

In [ ]:
# ── Swap spread (swap vs treasury) ──────────────────────────────────────────
# Requires SOFR swap data in the DataFrame — shows how swaps trade vs Treasuries
swap_cols = {t: c for t, c in {
    '2Y': 'ICERATES1100USD2Y', '5Y': 'ICERATES1100USD5Y',
    '10Y': 'ICERATES1100USD10Y', '30Y': 'ICERATES1100USD30Y'
}.items() if c in df.columns}

if swap_cols:
    rows = []
    for tenor, swap_col in swap_cols.items():
        tsy_col = TENOR_COLS.get(tenor)
        if tsy_col and tsy_col in df.columns:
            ss_series = fi.swap_spread_series(
                df.rename(columns={swap_col: tenor}),
                df.rename(columns={tsy_col: tenor}),
                tenor
            )
            curr = float(ss_series.iloc[-1]) if len(ss_series) > 0 else None
            z    = fi.zscore_current(ss_series, 252) if len(ss_series) > 252 else None
            rows.append({'Tenor': tenor, 'Swap Spread (bps)': round(curr,1) if curr else None,
                         'Z-score': round(z,2) if z else None})
    print(pd.DataFrame(rows).set_index('Tenor'))
else:
    print('SOFR swap data not available — check FRED fetch')

---
## 5. Spread Options

In [ ]:
# ── Price a spread option on the 2s10s spread ───────────────────────────────
t1, t2 = '2Y', '10Y'
spread_series = (get_rate_series(df, t2) - get_rate_series(df, t1)).dropna() * 100  # bps

setup = fi.spread_option_setup(
    spread_series,
    expiry_months = 3,
    strike_type   = 'ATM',
    vol_window    = 63,
)

print(f'2s10s Spread Option Setup:')
for k, v in setup.items():
    print(f'  {k}: {v}')

In [ ]:
# ── Expected return analysis for the bought call ──────────────────────────
exp_ret = fi.spread_option_expected_return(
    forward_spread = setup['Spread (bps)'],
    strike         = setup['Strike (bps)'],
    vol_bps_pa     = setup['Hist Vol (bps/yr)'],
    expiry_years   = setup['Expiry (yrs)'],
    premium_paid   = setup['Call Price (bps)'],
    option_type    = 'call',
)
print('Expected Return (bought call):')
for k, v in exp_ret.items():
    print(f'  {k}: {v}')

In [ ]:
# ── Screen all spreads for option opportunities ────────────────────────────
spread_dict = {}
for t1, t2 in fi.SPREAD_PAIRS:
    s1 = get_rate_series(df, t1)
    s2 = get_rate_series(df, t2)
    if len(s1) > 100 and len(s2) > 100:
        spread_dict[f'{t1}/{t2}'] = (s2 - s1).dropna() * 100  # bps

screen = fi.spread_option_screen(spread_dict, expiry_months=3)
display(screen)

---
## 6. Wedge Analysis
Forward rates vs spot — where is the best carry-adjusted trade?

In [ ]:
# ── Wedge grid (forward start × tail tenor) ────────────────────────────────
tenor_list = sorted([(fi.TENOR_YEARS[t], curve[t]) for t in curve if t in fi.TENOR_YEARS])
curve_tenors = [x[0] for x in tenor_list]
curve_rates  = [x[1] for x in tenor_list]

w_grid = fi.wedge_grid(curve_tenors, curve_rates)
print('Wedge Grid (bps) — rows=forward start, cols=tail tenor')
fi.style_returns_table(w_grid)

In [ ]:
# ── Ranked wedge trades (best risk-adjusted carry) ─────────────────────────
tenor_col_map = {t: TENOR_COLS[t] for t in TENOR_COLS if TENOR_COLS[t] in df.columns}

top_wedges = fi.run_wedge_analysis(
    curve_tenors  = curve_tenors,
    curve_rates   = curve_rates,
    rate_df       = df.rename(columns={v: k for k, v in TENOR_COLS.items()}),
    tenor_col_map = {t: t for t in fi.TENOR_YEARS if t in curve},
    vol_window    = 63,
    top_n         = 10,
)
display(top_wedges)

In [ ]:
# ── Specific wedge (e.g. 2Y into a 5Y swap = 2Y5Y) ─────────────────────────
FORWARD_START = 2.0  # ← change this
TAIL_TENOR    = 5.0  # ← change this

w = fi.wedge(curve_tenors, curve_rates, FORWARD_START, TAIL_TENOR)
fwd = fi.forward_swap_rate(curve_tenors, curve_rates, FORWARD_START, TAIL_TENOR)
spot = fi.interpolate_rate(curve_tenors, curve_rates, TAIL_TENOR)

print(f'{int(FORWARD_START)}Y{int(TAIL_TENOR)}Y Forward Swap Rate: {fwd:.3f}%')
print(f'{int(TAIL_TENOR)}Y Spot Swap Rate:          {spot:.3f}%')
print(f'Wedge:                        {w:.1f} bps')

regime = fi.vol_regime_check(get_rate_series(df, f'{int(TAIL_TENOR)}Y'))
print(f'Vol Regime: {regime}')

---
## 7. Swaptions
Price swaptions and analyse expected returns (buy cheap vol).

In [ ]:
# ── Price a swaption ──────────────────────────────────────────────────────
# Edit these:
EXPIRY_YRS  = 1.0     # option expiry
TAIL_YRS    = 10.0    # underlying swap tenor
STRIKE      = curve.get('10Y', 4.5)  # ATM = current 10Y rate
IMPL_VOL    = 80.0    # implied normal vol in bps/year
FWD_RATE    = fi.forward_swap_rate(curve_tenors, curve_rates, EXPIRY_YRS, TAIL_YRS)

payer = fi.bachelier_swaption(FWD_RATE, STRIKE, IMPL_VOL, EXPIRY_YRS, TAIL_YRS, 'payer')
rcvr  = fi.bachelier_swaption(FWD_RATE, STRIKE, IMPL_VOL, EXPIRY_YRS, TAIL_YRS, 'receiver')
greeks = fi.swaption_greeks(FWD_RATE, STRIKE, IMPL_VOL, EXPIRY_YRS, TAIL_YRS, 'payer')

print(f'{EXPIRY_YRS:.0f}Y{TAIL_YRS:.0f}Y Swaption (ATM, Bachelier):')
print(f'  Forward Swap Rate: {FWD_RATE:.3f}%')
print(f'  Strike:            {STRIKE:.3f}%')
print(f'  Impl Vol:          {IMPL_VOL:.0f} bps/yr')
print(f'  Payer premium:     {payer:.2f} bps')
print(f'  Receiver premium:  {rcvr:.2f} bps')
print(f'  Put-Call parity:   {payer-rcvr:.2f} bps (should ≈ 0 for ATM)')
print(f'  Greeks: {greeks}')

In [ ]:
# ── Expected return: what if realised vol ≠ implied vol? ──────────────────
REAL_VOL = 90.0  # ← your realised vol estimate

exp = fi.swaption_expected_return(
    forward_rate    = FWD_RATE,
    strike          = STRIKE,
    vol_normal_bps  = IMPL_VOL,
    expiry_years    = EXPIRY_YRS,
    tail_years      = TAIL_YRS,
    realised_vol_bps= REAL_VOL,
    swaption_type   = 'payer',
)
print('Expected Return Analysis (bought payer):')
for k, v in exp.items():
    print(f'  {k}: {v}')

In [ ]:
# ── Vol smile: visualise Bachelier prices across strikes ───────────────────
strikes = np.linspace(FWD_RATE - 1.0, FWD_RATE + 1.0, 40)
payer_prices  = [fi.bachelier_swaption(FWD_RATE, K, IMPL_VOL, EXPIRY_YRS, TAIL_YRS, 'payer')  for K in strikes]
rcvr_prices   = [fi.bachelier_swaption(FWD_RATE, K, IMPL_VOL, EXPIRY_YRS, TAIL_YRS, 'receiver') for K in strikes]

fig = go.Figure()
fig.add_trace(go.Scatter(x=strikes, y=payer_prices, name='Payer', line=dict(color='#ef5350')))
fig.add_trace(go.Scatter(x=strikes, y=rcvr_prices,  name='Receiver', line=dict(color='#42a5f5')))
fig.add_vline(x=FWD_RATE, line_dash='dash', line_color='white', annotation_text='ATM')
fig.update_layout(template='plotly_dark', title=f'{EXPIRY_YRS:.0f}Y{TAIL_YRS:.0f}Y Swaption Premium (bps)',
                  xaxis_title='Strike (%)', yaxis_title='Premium (bps)')
fig.show()

---
## 8. Portfolio Analytics
Beta, efficient frontier, Sharpe.

In [ ]:
# ── Rolling beta: 10Y vs 2Y ───────────────────────────────────────────────
s_2y  = get_rate_series(df, '2Y')
s_10y = get_rate_series(df, '10Y')
aligned = pd.concat([s_2y, s_10y], axis=1).dropna()
aligned.columns = ['2Y', '10Y']

beta = fi.rolling_beta(aligned['10Y'], aligned['2Y'], window=252)

reg = fi.regression_summary(aligned['10Y'], aligned['2Y'])
print('10Y vs 2Y Regression (1Y):')
for k, v in reg.items(): print(f'  {k}: {v}')

fig = go.Figure(go.Scatter(x=beta.index, y=beta.values, line=dict(color='#ab47bc')))
fig.add_hline(y=1.0, line_dash='dash', line_color='grey')
fig.update_layout(template='plotly_dark', title='Rolling 1Y Beta: 10Y vs 2Y',
                  xaxis_title='', yaxis_title='Beta')
fig.show()

In [ ]:
# ── Sharpe ratio table across tenors ──────────────────────────────────────
# Daily P&L proxy: position return = -DV01 × yield_change
# For simplicity, use daily yield changes as "returns"
available_tenors = [t for t in ['2Y','5Y','10Y','30Y'] if t in rate_df.columns]
returns_df = rate_df[available_tenors].diff().dropna() * -100  # bps P&L for a receiver
returns_df.columns = [f'Receive {t}' for t in available_tenors]

sharpe_df = fi.sharpe_table(returns_df)
display(sharpe_df)

In [ ]:
# ── Efficient frontier ─────────────────────────────────────────────────────
frontier = fi.efficient_frontier(returns_df.tail(504), n_portfolios=100, risk_free_rate=0)
max_sr   = fi.max_sharpe_portfolio(returns_df.tail(504))
min_var  = fi.min_variance_portfolio(returns_df.tail(504))

if not frontier.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=frontier['Vol (ann)'], y=frontier['Return (ann)'],
        mode='lines', name='Frontier', line=dict(color='#4fc3f7')
    ))
    if max_sr:
        fig.add_trace(go.Scatter(
            x=[max_sr['Volatility (ann)']], y=[max_sr['Expected Return (ann)']],
            mode='markers', marker=dict(size=12, color='#ffca28', symbol='star'),
            name=f'Max Sharpe ({max_sr["Sharpe"]:.2f})'
        ))
    if min_var:
        fig.add_trace(go.Scatter(
            x=[min_var['Volatility (ann)']], y=[min_var['Expected Return (ann)']],
            mode='markers', marker=dict(size=12, color='#ef5350', symbol='diamond'),
            name='Min Variance'
        ))
    fig.update_layout(template='plotly_dark', title='Efficient Frontier (Receive Fixed Swaps)',
                      xaxis_title='Annualised Vol (bps)', yaxis_title='Annualised Return (bps)')
    fig.show()

print('Max Sharpe portfolio:')
for k, v in max_sr.items(): print(f'  {k}: {v}')

In [ ]:
# ── Annual move indicator ──────────────────────────────────────────────────
annual_moves = fi.annual_move_indicator(rate_df)
display(annual_moves)

---
## 9. Weekly Update Tables (OneNote style)

In [ ]:
# ── Expected return table ──────────────────────────────────────────────────
exp_ret_table = fi.expected_return_table(
    rates          = curve,
    overnight_rate = on_rate,
    holding_months = 1,
    annualise      = True,
)
print('Expected Return (ann bps):')
fi.style_returns_table(exp_ret_table[['Carry','Rolldown','Total']])

In [ ]:
# ── Sharpe ratio table ────────────────────────────────────────────────────
rate_series_dict = {t: get_rate_series(df, t) for t in TENOR_COLS.keys()}

sharpe_tbl = fi.sharpe_table_from_rates(
    rates            = curve,
    overnight_rate   = on_rate,
    rate_series_dict = rate_series_dict,
    holding_months   = 1,
)
display(sharpe_tbl)

In [ ]:
# ── Z-score summary ────────────────────────────────────────────────────────
z_summary = fi.zscore_table(rate_df.ffill(limit=3).dropna(how='all'))
fi.style_zscore_table(z_summary)

In [ ]:
# ── Full weekly report (all tables) ──────────────────────────────────────
usd_rate_series = {t: get_rate_series(df, t) for t in curve.keys()}

report = fi.generate_weekly_report(
    rate_dict        = {'USD': curve},
    overnight_dict   = {'USD': on_rate},
    rate_series_dict = {'USD': usd_rate_series},
    currencies       = ['USD'],
    include_wedges   = True,
    include_spread_options = True,
)

for table_name, table_df in report.items():
    if not table_df.empty:
        print(f'\n── {table_name} ──')
        display(table_df)

---
## 10. Custom Analysis
Free-form cell — analyse any instrument or time period.

In [ ]:
# ── Fully custom: enter any trade here ────────────────────────────────────
# Example: 5s30s butterfly carry + z-score

W1, BELLY, W2 = '5Y', '10Y', '30Y'

s_w1   = get_rate_series(df, W1)
s_belly = get_rate_series(df, BELLY)
s_w2   = get_rate_series(df, W2)

fly_s = (s_belly - 0.5*s_w1 - 0.5*s_w2).dropna() * 100  # bps

z_fly  = fi.zscore(fly_s, 252)
cr_fly = fi.snapshot_carry_rolldown(curve, on_rate, 'fly', W1, BELLY, W2, holding_months=1)
summary = fi.summary_stats(fly_s)

print(f'{W1}/{BELLY}/{W2} Butterfly')
print(f'  Current: {float(fly_s.iloc[-1]):.1f} bps')
print(f'  Z-score: {float(z_fly.iloc[-1]):.2f}')
print(f'  Carry+Rolldown: {cr_fly["total"]:.2f} bps/month')
display(summary)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=[f'{W1}/{BELLY}/{W2} Fly (bps)', 'Z-score'])
fig.add_trace(go.Scatter(x=fly_s.index, y=fly_s.values, line=dict(color='#66bb6a')), row=1, col=1)
fig.add_trace(go.Scatter(x=z_fly.index, y=z_fly.values, line=dict(color='#ffa726')), row=2, col=1)
fig.add_hline(y=1.5, line_dash='dash', line_color='lime', row=2, col=1)
fig.add_hline(y=-1.5, line_dash='dash', line_color='red', row=2, col=1)
fig.update_layout(template='plotly_dark', height=500, showlegend=False)
fig.show()